# D172 — CTE Exercises: Seller Analytics

This hands-on notebook continues the Olist analysis from D170. Every problem focuses on sellers and must be solved with one or more **common table expressions (CTEs)**.

## Learning goals

- define a CTE with `WITH`;
- aggregate inside a CTE and filter its result outside;
- join a CTE to a regular table;
- define multiple CTEs in one statement;
- build chained CTEs where one CTE uses another; and
- make a long seller-analysis query easier to read and verify.

Do not use window functions in this notebook. Do not create permanent tables or views.


## 1. Connect to MySQL

The defaults match the D16 and D170 classroom setup. Environment variables can override them.


In [ ]:
import os
import mysql.connector
from mysql.connector import Error

connection = mysql.connector.connect(
    host=os.environ.get("MYSQL_HOSTNAME", "127.0.0.1"),
    port=int(os.environ.get("MYSQL_PORT", "3306")),
    user=os.environ.get("MYSQL_USERNAME", "root"),
    password=os.environ.get("MYSQL_PASSWORD", "root"),
    database=os.environ.get("MYSQL_DATABASE", "olist_import_lab"),
)

print("Connected:", connection.is_connected())
print("MySQL version:", connection.server_info)


## 2. Query helpers

`execute_sql` sends SQL to MySQL and uses `print_rows` to display a maximum of 25 rows.


In [ ]:
def print_rows(columns, rows):
    if not rows:
        print("No rows returned.")
        return

    text_rows = [["NULL" if value is None else str(value) for value in row]
                 for row in rows]
    widths = [len(str(column)) for column in columns]
    for row in text_rows:
        widths = [max(width, len(value)) for width, value in zip(widths, row)]

    print(" | ".join(str(column).ljust(width)
                     for column, width in zip(columns, widths)))
    print("-+-".join("-" * width for width in widths))
    for row in text_rows:
        print(" | ".join(value.ljust(width)
                         for value, width in zip(row, widths)))


def execute_sql(sql, params=None, max_rows=25):
    cursor = connection.cursor()
    try:
        cursor.execute(sql, params or ())
        if not cursor.with_rows:
            connection.commit()
            print(f"Statement completed. Affected rows: {cursor.rowcount:,}")
            return cursor.rowcount

        columns = [item[0] for item in cursor.description]
        rows = cursor.fetchmany(max_rows + 1)
        visible_rows = rows[:max_rows]
        print_rows(columns, visible_rows)
        if len(rows) > max_rows:
            print(f"... showing the first {max_rows} rows")
        return visible_rows
    except Error:
        connection.rollback()
        raise
    finally:
        cursor.close()


## 3. Seller data and join paths

| Table | Relevant columns |
|---|---|
| `olist_sellers` | `seller_id`, `seller_city`, `seller_state` |
| `olist_order_items` | `order_id`, `product_id`, `seller_id`, `price`, `freight_value` |
| `olist_orders` | `order_id`, `customer_id`, `order_status`, `order_purchase_timestamp` |
| `olist_customers` | `customer_id`, `customer_state` |
| `olist_products` | `product_id`, `product_category_name` |
| `product_category_translation` | `product_category_name`, `product_category_name_english` |
| `olist_order_reviews` | `order_id`, `review_score` |

Useful paths:

- `sellers → order_items → orders`
- `sellers → order_items → products → category_translation`
- `sellers → order_items → orders → customers`
- `sellers → order_items → reviews` through `order_id`

One order can contain several item rows. One seller can serve many orders. Choose the correct counting level and use `COUNT(DISTINCT ...)` when the business question asks for orders rather than items.


In [ ]:
execute_sql("""
SELECT 'sellers' AS table_name, COUNT(*) AS row_count FROM olist_sellers
UNION ALL
SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL
SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL
SELECT 'products', COUNT(*) FROM olist_products
UNION ALL
SELECT 'reviews', COUNT(*) FROM olist_order_reviews
""")


## 4. CTE reminder

A CTE is a named query result available only to the statement that immediately follows it.

```sql
WITH cte_name AS (
    SELECT ...
    FROM ...
)
SELECT ...
FROM cte_name;
```

For multiple CTEs, write `WITH` once and separate definitions with commas. A later CTE may reference an earlier one.

Each exercise below has an empty code cell for your answer.


## Exercise 1: High-revenue sellers

The marketplace team wants a shortlist of sellers with large lifetime item revenue.

**Task:** Create a CTE named `seller_sales` that groups `olist_order_items` by seller. Calculate item count, distinct order count, and total item revenue. In the outer query, return sellers whose revenue is at least 100,000.

**Must use:** one aggregate CTE, `COUNT`, `COUNT(DISTINCT ...)`, `SUM`, and an outer `WHERE`.

**Expected result:** Columns `seller_id`, `item_count`, `order_count`, and `item_revenue`, ordered by revenue descending. Round revenue to two decimals in the final output.


## Exercise 2: Seller performance with location

Regional management wants high-performing sellers together with their location.

**Task:** Build a CTE that calculates total item revenue and items sold per seller. Join that CTE to `olist_sellers`, then return the top 15 sellers with their city and state.

**Must use:** one aggregate CTE, an outer `INNER JOIN`, `ORDER BY`, and `LIMIT`.

**Expected result:** Exactly 15 rows with columns `seller_id`, `seller_city`, `seller_state`, `items_sold`, and `item_revenue`; highest revenue first.


## Exercise 3: Delivered sales by seller

Operations wants seller totals based only on successfully delivered orders.

**Task:** Create a CTE called `delivered_items` by joining orders and order items and filtering `order_status = 'delivered'`. In the outer query, group the CTE rows by seller and calculate delivered order count, delivered item count, and item revenue. Return sellers with at least 100 distinct delivered orders.

**Must use:** a join and `WHERE` inside the CTE; aggregation and `HAVING` outside it.

**Expected result:** Columns `seller_id`, `delivered_orders`, `delivered_items`, and `item_revenue`; all rows meet the 100-order threshold and are sorted by delivered orders descending.


## Exercise 4: Seller totals compared with state totals

Management wants to understand each seller’s contribution to the seller state where the seller is registered.

**Task:** Define two CTEs. `seller_sales` should calculate revenue per seller and retain seller state. `state_sales` should aggregate `seller_sales` into revenue per state. Join the two CTEs and return sellers with at least 100,000 in revenue, together with their state revenue.

**Must use:** two CTEs, where the second reads from the first; an outer join between the CTEs.

**Expected result:** Columns `seller_id`, `seller_state`, `seller_revenue`, and `state_revenue`, ordered by seller revenue descending. Monetary values should be rounded to two decimals.


## Exercise 5: Seller category diversity

The merchandising team wants sellers that operate across many product categories.

**Task:** In a CTE, join order items, products, and category translation. Group by seller and calculate distinct English category count, item count, and total revenue. Exclude null English category names before aggregation. In the outer query, keep sellers with at least 20 distinct categories.

**Must use:** a three-table CTE, `WHERE`, `COUNT(DISTINCT ...)`, `SUM`, and an outer filter.

**Expected result:** Columns `seller_id`, `category_count`, `item_count`, and `item_revenue`; category count must be at least 20. Sort by category count descending, then revenue descending.


## Exercise 6: Seller review quality

Customer experience wants to find established sellers with strong review scores.

**Task:** Create a CTE that joins order items and reviews by `order_id`. Group by seller and calculate distinct reviewed orders, minimum score, maximum score, and average score. The outer query should keep sellers having at least 100 reviewed orders and an average score of at least 4.0.

**Must use:** one aggregate CTE, `COUNT(DISTINCT ...)`, `MIN`, `MAX`, `AVG`, and outer `WHERE` conditions.

**Expected result:** Columns `seller_id`, `reviewed_orders`, `minimum_score`, `maximum_score`, and `average_score`. Round the average to two decimals and sort it descending, using reviewed orders as the second sort.


## Exercise 7: Local versus interstate seller activity

Logistics wants to compare sales where the seller and customer are in the same state against sales crossing state boundaries.

**Task:** Create a detail CTE joining sellers, order items, orders, and customers. Include seller ID, order ID, price, seller state, and customer state. In the outer query, group by seller and calculate distinct orders, same-state item revenue, and different-state item revenue using conditional `SUM(CASE ...)`. Keep sellers with at least 100 distinct orders.

**Must use:** a four-table CTE, outer aggregation, `CASE`, `SUM`, and `HAVING`.

**Expected result:** Columns `seller_id`, `order_count`, `same_state_revenue`, and `different_state_revenue`; ordered by total of the two revenue columns descending.


## Exercise 8: Monthly seller activity summary

The sales team wants to review seller performance during 2018 without mixing date filtering into the final report query.

**Task:** Create a CTE containing 2018 order items joined to their orders. Use a half-open date range from `2018-01-01` through, but not including, `2019-01-01`. In the outer query, group by seller and purchase month, calculate distinct orders and item revenue, and retain seller-month groups with at least 50 orders.

**Must use:** a date-filtered CTE, `DATE_FORMAT`, `COUNT(DISTINCT ...)`, `SUM`, `GROUP BY`, and `HAVING`.

**Expected result:** Columns `seller_id`, `purchase_month`, `order_count`, and `item_revenue`; ordered chronologically by month and then by revenue descending.


## Exercise 9: Above-average seller revenue

Leadership wants sellers whose lifetime revenue is above the average revenue of all sellers.

**Task:** Use three chained CTEs. First calculate revenue per seller. Second calculate the overall average of those seller revenues. Third select or combine the information needed for the final result. Return only sellers above the overall seller average.

**Must use:** at least two aggregate levels in chained CTEs and no subquery in the final `WHERE` clause.

**Expected result:** Columns `seller_id`, `seller_revenue`, and `average_seller_revenue`. Every seller revenue must be greater than the displayed average. Sort by seller revenue descending.


## Exercise 10: Seller performance dashboard

The marketplace director wants one compact report combining seller sales, location, and customer satisfaction.

**Task:** Define separate CTEs for seller sales and seller reviews. Seller sales must include item count, distinct order count, total item revenue, and total freight. Seller reviews must include distinct reviewed orders and average review score. Join both CTEs to `olist_sellers`. Keep sellers with at least 200 orders and at least 100 reviewed orders, then return the top 20 by item revenue.

**Must use:** multiple independent CTEs, aggregate functions, joins between CTE results and a base table, outer filtering, and `LIMIT`. Do not use a window function.

**Expected result:** Up to 20 rows with columns `seller_id`, `seller_city`, `seller_state`, `item_count`, `order_count`, `item_revenue`, `freight_total`, `reviewed_orders`, and `average_review_score`; highest item revenue first.

**Important:** Aggregate sales and reviews separately before joining them. Joining raw item rows directly to raw review rows can multiply records and inflate totals.


## Review checklist

- Start the statement with one `WITH` keyword.
- Give every CTE a meaningful name.
- Separate multiple CTE definitions with commas.
- Confirm whether the problem counts item rows or distinct orders.
- Aggregate tables at their correct level before joining aggregate results.
- Use `WHERE` for detail rows and `HAVING` for grouped results.
- Round displayed monetary and average values to two decimals.
- Do not use window functions in any answer.


## Close the connection

Run this after completing the exercises.


In [ ]:
if connection.is_connected():
    connection.close()
print("MySQL connection closed.")
